In [1]:
!apt-get update -qq
!apt-get install -y flex bison gcc

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
gcc is already the newest version (4:11.2.0-1ubuntu1).
gcc set to manually installed.
The following additional packages will be installed:
  libfl-dev libfl2
Suggested packages:
  bison-doc flex-doc
The following NEW packages will be installed:
  bison flex libfl-dev libfl2
0 upgraded, 4 newly installed, 0 to remove and 137 not upgraded.
Need to get 1,072 kB of archives.
After this operation, 3,667 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 flex amd64 2.6.4-8build2 [307 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/main amd64 bison amd64 2:3.8.2+dfsg-1build1 [748 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/main amd64 libfl2 amd64 2.6.4-8build2

In [9]:
%%writefile tac.l
%{
#include "tac.tab.h"
#include <string.h>
#include <stdlib.h>
%}

%option noyywrap

%%

[a-zA-Z][a-zA-Z0-9]* {
    yylval.str = strdup(yytext);
    return ID;
}

[0-9]+ {
    yylval.str = strdup(yytext);
    return NUM;
}

[ \t\n]+ {
    /* Ignore whitespace */
}

. {
    return yytext[0];
}

%%

Overwriting tac.l


In [10]:
%%writefile tac.y
%{
#include <stdio.h>
#include <stdlib.h>
#include <string.h>

int tempCount = 1;

void printTAC(char *result, char *op1, char *operator, char *op2)
{
    printf("%s = %s %s %s\n", result, op1, operator, op2);
}

void printAssign(char *var, char *val)
{
    printf("%s = %s\n", var, val);
}

int yylex(void);
int yyerror(const char *s);
%}

%union {
    char *str;
}

%token <str> ID NUM
%type <str> expr

%left '+' '-'
%left '*' '/'

%%

stmt:
      ID '=' expr
      {
          printAssign($1, $3);
      }
      ;

expr:
      expr '+' expr
      {
          char temp[20];
          sprintf(temp, "t%d", tempCount++);
          printTAC(temp, $1, "+", $3);
          $$ = strdup(temp);
      }

    | expr '-' expr
      {
          char temp[20];
          sprintf(temp, "t%d", tempCount++);
          printTAC(temp, $1, "-", $3);
          $$ = strdup(temp);
      }

    | expr '*' expr
      {
          char temp[20];
          sprintf(temp, "t%d", tempCount++);
          printTAC(temp, $1, "*", $3);
          $$ = strdup(temp);
      }

    | expr '/' expr
      {
          char temp[20];
          sprintf(temp, "t%d", tempCount++);
          printTAC(temp, $1, "/", $3);
          $$ = strdup(temp);
      }

    | ID
      {
          $$ = $1;
      }

    | NUM
      {
          $$ = $1;
      }
      ;

%%

int main(void)
{
    printf("Enter the expression:\n");
    yyparse();
    return 0;
}

int yyerror(const char *s)
{
    printf("Error: %s\n", s);
    return 0;
}

Overwriting tac.y


In [4]:
!bison -d tac.y

In [5]:
!flex tac.l

In [6]:
!gcc tac.tab.c lex.yy.c -o tac

/usr/bin/ld: /tmp/ccpVibd2.o: in function `yylex':
lex.yy.c:(.text+0x4d1): undefined reference to `yywrap'
/usr/bin/ld: /tmp/ccpVibd2.o: in function `input':
lex.yy.c:(.text+0x10e7): undefined reference to `yywrap'
collect2: error: ld returned 1 exit status


In [7]:
!echo "a=b+c*d" | ./tac

/bin/bash: line 1: ./tac: No such file or directory


In [8]:
!echo "a=b+c" | ./tac

/bin/bash: line 1: ./tac: No such file or directory


In [11]:
!bison -d tac.y

In [12]:
!ls -l tac.tab.c tac.tab.h

-rw-r--r-- 1 root root 41502 Aug 11 15:41 tac.tab.c
-rw-r--r-- 1 root root  2648 Aug 11 15:41 tac.tab.h


In [13]:
!flex tac.l

In [14]:
!ls -l lex.yy.c

-rw-r--r-- 1 root root 44679 Aug 11 15:41 lex.yy.c


In [15]:
!gcc tac.tab.c lex.yy.c -o tac

In [16]:
!ls -l tac

-rwxr-xr-x 1 root root 32064 Aug 11 15:42 tac


In [17]:
!echo "a=b+c*d" | ./tac

Enter the expression:
t1 = c * d
t2 = b + t1
a = t2
